# 01 — Extraction des traits de côte de Saint-Louis (1984-2026)

Ce notebook tourne dans **Google Colab**. Il télécharge les images Landsat 5/7/8/9 et Sentinel-2 de Saint-Louis via Google Earth Engine, puis extrait automatiquement le trait de côte de chaque image avec **CoastSat**.

Toutes les données sont enregistrées dans ton Google Drive, dans `Mon Drive/erosion-cotiere-saint-louis-senegal/`.

**Sorties :**
- `data/processed/traits_de_cote_saint_louis.geojson` : un trait de côte par date (EPSG:32628)
- `outputs/figures/timelapse_saint_louis.gif` : animation des images prétraitées
- `outputs/figures/apercu_traits_de_cote.png` : carte de synthèse

**Avant de commencer :** dans Colab, menu *Exécution > Modifier le type d'exécution*, le processeur standard (CPU) suffit.

## 1. Paramètres à régler

In [ ]:
# Nom de ton projet Google Cloud enregistré dans Earth Engine
PROJET_GEE = "erosion-saint-louis"

# MODE_TEST = True : une seule année de Sentinel-2, pour valider la chaîne en quelques minutes
# MODE_TEST = False : série complète, découpée en 4 périodes (une session Colab par période)
MODE_TEST = False

# Période à traiter quand MODE_TEST = False :
#   "auto" : la première période dont le fichier manque sur le Drive (il suffit de
#            relancer « Tout exécuter » jusqu'à ce que toutes soient faites)
#   1 : 1984-1999 (Landsat 5 et 7)
#   2 : 2000-2014 (Landsat 5, 7 et 8)
#   3 : 2015-2020 (Landsat 8 et Sentinel-2)
#   4 : 2021-2026 (Sentinel-2 seul : une image tous les 5 jours à 10 m)
PERIODE = "auto"

## 2. Google Drive et dossiers du projet

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
ROOT = "/content/drive/MyDrive/erosion-cotiere-saint-louis-senegal"
for d in ["data/aoi", "data/processed", "outputs/figures", "outputs/cartes"]:
    os.makedirs(os.path.join(ROOT, d), exist_ok=True)
print("Dossier du projet :", ROOT)

## 3. Installation de CoastSat

In [ ]:
import importlib, subprocess, sys

# Bibliothèques nécessaires à CoastSat : on n'installe que celles qui manquent dans Colab
requis = {"ee": "earthengine-api", "geopandas": "geopandas", "skimage": "scikit-image",
          "sklearn": "scikit-learn", "astropy": "astropy", "imageio": "imageio",
          "pytz": "pytz", "osgeo": "gdal"}
manquants = [pip for mod, pip in requis.items() if importlib.util.find_spec(mod) is None]
if manquants:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *manquants], check=True)
print("Paquets installés :", manquants or "aucun, tout est déjà là")

# CoastSat (recloné à chaque session Colab, hors du Drive)
if not os.path.exists("/content/CoastSat"):
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/kvos/CoastSat.git", "/content/CoastSat"], check=True)
sys.path.append("/content/CoastSat")
# CoastSat cherche ses modèles de classification dans le dossier courant
os.chdir("/content/CoastSat")

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from pyproj import CRS
from coastsat import SDS_download, SDS_preprocess, SDS_shoreline, SDS_tools
print("CoastSat prêt")

## 4. Connexion à Google Earth Engine

Une fenêtre te demande de choisir ton compte Google : prends celui du projet `erosion-saint-louis`.

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project=PROJET_GEE)
print("Earth Engine connecté au projet", PROJET_GEE)

## 5. Zone d'étude

La zone couvre la ville de Saint-Louis, Guet Ndar et la Langue de Barbarie jusqu'au sud de la brèche ouverte en 2003. Elle est enregistrée dans `data/aoi/saint_louis_aoi.geojson` (créée automatiquement si elle n'existe pas encore).

In [ ]:
import json
fn_aoi = os.path.join(ROOT, "data", "aoi", "saint_louis_aoi.geojson")

if not os.path.exists(fn_aoi):
    aoi = {
        "type": "FeatureCollection",
        "features": [{
            "type": "Feature",
            "properties": {"nom": "Saint-Louis - Langue de Barbarie"},
            "geometry": {"type": "Polygon", "coordinates": [[
                [-16.535, 16.060], [-16.475, 16.060], [-16.475, 15.900],
                [-16.535, 15.900], [-16.535, 16.060]]]}
        }]
    }
    with open(fn_aoi, "w", encoding="utf-8") as f:
        json.dump(aoi, f, ensure_ascii=False, indent=2)

polygon = SDS_tools.polygon_from_geojson(fn_aoi)
polygon = SDS_tools.smallest_rectangle(polygon)

surface_km2 = gpd.GeoSeries.from_wkt(
    [f"POLYGON(({', '.join(f'{x} {y}' for x, y in polygon[0])}))"], crs=4326
).to_crs(32628).area.iloc[0] / 1e6
print(f"Surface de la zone d'étude : {surface_km2:.1f} km²")

## 6. Images disponibles

Cette cellule ne télécharge rien : elle compte seulement les images disponibles sur la période.

En mode complet, la série est découpée en quatre périodes, traitées chacune dans sa propre session Colab : si une session se coupe, on ne perd qu'une période. Chaque période produit son propre fichier dans `data/processed`.

In [ ]:
PERIODES = {
    1: (["1984-01-01", "2000-01-01"], ["L5", "L7"], "1984_1999"),
    2: (["2000-01-01", "2015-01-01"], ["L5", "L7", "L8"], "2000_2014"),
    3: (["2015-01-01", "2021-01-01"], ["L8", "S2"], "2015_2020"),
    4: (["2021-01-01", "2026-10-01"], ["S2"], "2021_2026"),
}

if MODE_TEST:
    dates, sat_list, sitename = ["2024-01-01", "2025-01-01"], ["S2"], "SAINT_LOUIS_TEST"
    suffixe = "_test"
else:
    if PERIODE == "auto":
        manquantes = [k for k, (_, _, e) in PERIODES.items()
                      if not os.path.exists(os.path.join(ROOT, "data", "processed",
                                                         f"traits_de_cote_saint_louis_p{k}_{e}.geojson"))]
        if not manquantes:
            raise SystemExit("Toutes les périodes sont déjà traitées : exécute seulement la section 15 (fusion).")
        PERIODE = manquantes[0]
        print(f"Mode automatique : période {PERIODE} (périodes restantes : {manquantes})")
    dates, sat_list, etiquette = PERIODES[PERIODE]
    sitename = f"SAINT_LOUIS_P{PERIODE}"
    suffixe = f"_p{PERIODE}_{etiquette}"
print(f"Traitement : {dates[0]} → {dates[1]}, satellites {sat_list}")

inputs = {
    "polygon": polygon,
    "dates": dates,
    "sat_list": sat_list,
    "sitename": sitename,
    # Images sur le disque temporaire de Colab (~100 Go) : trop lourdes pour le Drive
    "filepath": "/content/raw",
    # Landsat 7 après mai 2003 a des bandes manquantes (panne du SLC) : on les ignore
    "skip_L7_SLC": True,
}

SDS_download.check_images_available(inputs);

## 7. Téléchargement des images

Les images sont découpées sur la zone et enregistrées sur le **disque temporaire de Colab** (`/content/raw`), pas sur le Drive : environ 16 Mo par image Sentinel-2, soit plusieurs Go pour la série complète. Seuls les résultats (GeoJSON, GIF, PNG) sont enregistrés sur le Drive.

⚠️ Le disque de Colab est effacé à la fin de la session : si elle se coupe, il faut relancer le téléchargement. En mode complet, cette étape peut prendre plusieurs heures : garde l'onglet Colab ouvert.

Si les images sont déjà téléchargées dans la session en cours, exécute seulement la cellule suivante pour recharger les métadonnées.

In [ ]:
metadata = SDS_download.retrieve_images(inputs)

In [ ]:
# Si les images sont déjà présentes dans data/raw
metadata = SDS_download.get_metadata(inputs)

### 7 bis. Vérification des images

Contrôle que chaque image téléchargée est complète. Les images abîmées (téléchargement interrompu, par exemple après une déconnexion de Colab) sont mises de côté dans `_images_corrompues` et retirées de la liste. Relancer la cellule 7 les retéléchargera.

In [ ]:
# Vérification des images téléchargées : chaque image Sentinel-2 / Landsat doit être lisible
import shutil
from osgeo import gdal
gdal.UseExceptions()

def lisible(chemin):
    try:
        return os.path.exists(chemin) and gdal.Open(chemin) is not None
    except Exception:
        return False

dossier_site = os.path.join(inputs["filepath"], sitename)
dossier_ecarte = os.path.join(dossier_site, "_images_corrompues")
total_ecartees = 0

for satname in list(metadata.keys()):
    fp = SDS_tools.get_filepath(inputs, satname)
    garder = []
    for i, nom in enumerate(metadata[satname]["filenames"]):
        fichiers = SDS_tools.get_filenames(nom, fp, satname)
        if all(lisible(f) for f in fichiers):
            garder.append(i)
            continue
        # Image abîmée : on met de côté ses fichiers et sa fiche de métadonnées,
        # pour que la cellule 7 puisse la retélécharger plus tard
        os.makedirs(dossier_ecarte, exist_ok=True)
        fiche = os.path.join(dossier_site, satname, "meta",
                             nom.replace("_ms", "").replace(".tif", ".txt"))
        for f in fichiers + [fiche]:
            if os.path.exists(f):
                shutil.move(f, os.path.join(dossier_ecarte, os.path.basename(f)))
        total_ecartees += 1
    for cle in metadata[satname]:
        metadata[satname][cle] = [metadata[satname][cle][i] for i in garder]
    print(f"{satname} : {len(garder)} images valides")

print(f"{total_ecartees} image(s) incomplète(s) mise(s) de côté dans _images_corrompues")

## 8. Paramètres d'extraction

In [ ]:
settings = {
    # Généraux
    "cloud_thresh": 0.1,        # couverture nuageuse maximale acceptée (10 %)
    "dist_clouds": 300,         # distance aux nuages (m) où le trait de côte n'est pas tracé
    "output_epsg": 32628,       # WGS 84 / UTM zone 28N
    # Contrôle qualité (les fenêtres interactives ne fonctionnent pas dans Colab)
    "check_detection": False,
    "adjust_detection": False,
    "save_figure": False,
    # Détection
    "min_beach_area": 1000,     # surface minimale d'une plage (m²)
    "min_length_sl": 500,       # longueur minimale d'un trait de côte (m)
    "cloud_mask_issue": False,
    "sand_color": "default",    # essayer "bright" si le sable clair est mal classé
    "pan_off": False,
    "s2cloudless_prob": 40,
    "inputs": inputs,
}

## 9. Prétraitement et animation

Masquage des nuages, pansharpening, export en JPG daté, puis animation de la série (pratique pour visualiser la brèche).

Les figures de CoastSat sont fermées au fur et à mesure, et le GIF est construit image par image (150 images au maximum) : sinon, la mémoire de Colab sature avec les centaines d'images de la période 3.

In [ ]:
plt.ioff()

# CoastSat ne ferme pas ses figures matplotlib : avec des centaines d'images, la mémoire
# de Colab sature et la session plante. On ferme chaque figure juste après l'avoir enregistrée.
if not getattr(SDS_preprocess.create_jpg, "ferme_les_figures", False):
    _create_jpg_origine = SDS_preprocess.create_jpg
    def _create_jpg_sans_fuite(*args, **kwargs):
        _create_jpg_origine(*args, **kwargs)
        plt.close("all")
    _create_jpg_sans_fuite.ferme_les_figures = True
    SDS_preprocess.create_jpg = _create_jpg_sans_fuite

SDS_preprocess.save_jpg(metadata, settings, use_matplotlib=True)
plt.close("all")

# Animation GIF construite image par image (sans tout charger en mémoire),
# limitée à 150 images réparties sur la période et réduites à 900 px de large
import imageio.v2 as imageio
from PIL import Image

fp_images = os.path.join(inputs["filepath"], sitename, "jpg_files", "preprocessed")
fichiers = sorted(f for f in os.listdir(fp_images) if f.lower().endswith(".jpg"))
print(f"{len(fichiers)} images prétraitées")

if fichiers:
    choix = np.unique(np.linspace(0, len(fichiers) - 1, min(150, len(fichiers))).astype(int))
    largeur = 900
    w0, h0 = Image.open(os.path.join(fp_images, fichiers[0])).size
    taille = (largeur, int(h0 * largeur / w0))
    fn_anim = os.path.join(ROOT, "outputs", "figures", f"timelapse_{sitename.lower()}.gif")
    with imageio.get_writer(fn_anim, mode="I", duration=250, loop=0) as gif:  # 250 ms par image
        for k in choix:
            with Image.open(os.path.join(fp_images, fichiers[k])) as im:
                gif.append_data(np.array(im.convert("RGB").resize(taille)))
    print(f"Animation enregistrée ({len(choix)} images) : {fn_anim}")

## 10. Trait de côte de référence

Le trait de référence sert à éliminer les fausses détections : nuages au large, berges du fleuve et de la lagune. CoastSat ne garde que les contours situés à moins de `max_dist_ref` mètres de ce trait.

Le fichier `data/aoi/trait_de_cote_reference.geojson` est **généré automatiquement** à partir d'un premier passage sans filtre (section 14) : pour chaque bande nord-sud de 25 m, on retient la position la plus à l'ouest détectée sur au moins la moitié des dates, c'est-à-dire la plage côté océan. Il peut être vérifié ou retouché dans QGIS.

⚠️ La Langue de Barbarie a deux rives sableuses (océan et fleuve) : le trait côté océan évite que CoastSat capte la rive du fleuve. Le trait est coupé au niveau de l'embouchure. Sur les images anciennes, où la côte a bougé de plusieurs kilomètres au niveau de la brèche, ce secteur sera traité séparément.

Si le fichier n'existe pas, l'extraction se fait sans filtre.

In [ ]:
fn_ref = os.path.join(ROOT, "data", "aoi", "trait_de_cote_reference.geojson")

if os.path.exists(fn_ref):
    ref = gpd.read_file(fn_ref).to_crs(settings["output_epsg"])
    # Densification tous les 5 m : CoastSat construit sa zone tampon à partir des points du trait
    points = []
    for geom in ref.geometry:
        lignes = getattr(geom, "geoms", [geom])
        for ligne in lignes:
            distances = np.arange(0, ligne.length, 5)
            points += [ligne.interpolate(d).coords[0][:2] for d in distances]
    settings["reference_shoreline"] = np.array(points)
    # Distance maximale (m) au trait de référence (tracé sur 2024). 300 m couvrent les déplacements
    # observés depuis 1984 (jusqu'à ~260 m). Au-delà, CoastSat inclut le fleuve et la ville dans
    # son calcul de seuil sable/eau, et la plage du nord n'est plus détectée.
    settings["max_dist_ref"] = 150 if MODE_TEST else 300
    print(f"Trait de référence chargé : {len(points)} points")
else:
    print("Pas de trait de référence : extraction sans filtre")

## 11. Extraction des traits de côte

In [ ]:
output = SDS_shoreline.extract_shorelines(metadata, settings)
plt.close("all")

# Nettoyage : doublons d'une même date et géoréférencement imprécis (> 10 m)
output = SDS_tools.remove_duplicates(output)
output = SDS_tools.remove_inaccurate_georef(output, 10)

print(f"{len(output['dates'])} traits de côte extraits")

## 12. Export en GeoJSON

Avant l'export, deux filtres ne gardent que la plage côté océan : sur chaque date, le contour le plus à l'ouest dans chaque bande nord-sud de 25 m, puis un contrôle par rapport à la position médiane de la plage sur la période. Ils éliminent la rive du fleuve, que la tolérance de 300 m peut capter là où la Langue de Barbarie est étroite. Aucune limite fixe n'est imposée côté terre : un recul réel de plus de 200 m depuis 1984 doit rester visible.

Format lisible dans QGIS, ArcGIS Pro (pour DSAS) et importable dans PostGIS.

In [ ]:
from shapely.geometry import LineString, MultiLineString
from scipy.spatial import cKDTree

def filtre_cote_ocean(points, ref=None, pas_y=25, largeur=60, recul_max=200):
    # Ne garde que le contour côté océan : sur une même date, dans chaque bande nord-sud
    # de 25 m, la plage océanique est le contour le plus à l'ouest. Les points situés à
    # plus de 60 m à l'est de ce contour (rive du fleuve, lagune) sont retirés.
    # Option : retire aussi les points à plus de 200 m à l'est (côté terre) du trait de référence.
    points = np.asarray(points)[:, :2]
    if len(points) == 0:
        return points
    bandes = np.floor(points[:, 1] / pas_y).astype(int)
    x_ouest = {b: points[bandes == b, 0].min() for b in np.unique(bandes)}
    garder = points[:, 0] - np.array([x_ouest[b] for b in bandes]) <= largeur
    if ref is not None and len(ref):
        _, idx = cKDTree(ref).query(points)
        garder &= (points[:, 0] - ref[idx, 0]) <= recul_max
    return points[garder]

def filtre_mediane_periode(liste_points, pas_y=25, ecart_max=150):
    # Second contrôle, sur toutes les dates de la période : dans chaque bande nord-sud,
    # on calcule la position médiane de la plage, et on retire les points situés à plus de
    # 150 m à l'est de cette médiane (rive du fleuve captée quand la plage était sous les nuages)
    ouest = {}
    for pts in liste_points:
        if len(pts) == 0:
            continue
        bandes = np.floor(pts[:, 1] / pas_y).astype(int)
        for b in np.unique(bandes):
            ouest.setdefault(b, []).append(pts[bandes == b, 0].min())
    mediane = {b: np.median(v) for b, v in ouest.items()}
    resultat = []
    for pts in liste_points:
        if len(pts) == 0:
            resultat.append(pts)
            continue
        bandes = np.floor(pts[:, 1] / pas_y).astype(int)
        resultat.append(pts[pts[:, 0] - np.array([mediane[b] for b in bandes]) <= ecart_max])
    return resultat


def decouper_trait(points, ecart_max=50, n_min=5):
    # CoastSat renvoie des points : on les relie seulement entre voisins proches (< 50 m),
    # pour éviter les longues lignes droites entre deux contours éloignés
    points = np.asarray(points)[:, :2]
    if len(points) < 2:
        return None
    sauts = np.where(np.hypot(*np.diff(points, axis=0).T) > ecart_max)[0] + 1
    morceaux = [m for m in np.split(points, sauts) if len(m) >= n_min]
    return MultiLineString([LineString(m) for m in morceaux]) if morceaux else None

# Filtres côté océan, date par date puis sur toute la période
points_ocean = [filtre_cote_ocean(p) for p in output["shorelines"]]
points_ocean = filtre_mediane_periode(points_ocean)

lignes = []
for i, pts in enumerate(points_ocean):
    geom = decouper_trait(pts)
    if geom is None:
        continue
    lignes.append({
        "date": output["dates"][i].strftime("%Y-%m-%d %H:%M:%S"),
        "satname": output["satname"][i],
        "geoaccuracy": output["geoaccuracy"][i],
        "cloud_cover": output["cloud_cover"][i],
        "geometry": geom,
    })
if not lignes:
    raise Exception("Aucun trait de côte extrait")

gdf = gpd.GeoDataFrame(lignes, crs=CRS(settings["output_epsg"]))
gdf["annee"] = gdf["date"].str[:4].astype(int)
gdf["mois"] = gdf["date"].str[5:7].astype(int)

fn_out = os.path.join(ROOT, "data", "processed", f"traits_de_cote_saint_louis{suffixe}.geojson")
gdf.to_file(fn_out, driver="GeoJSON", encoding="utf-8")
print(f"{len(gdf)} traits de côte exportés : {fn_out}")
gdf.drop(columns="geometry").head()

## 13. Aperçu

In [ ]:
plt.ion()
champ, titre = ("mois", "Mois") if MODE_TEST else ("annee", "Année")
fig, ax = plt.subplots(figsize=(8, 12))
gdf.plot(ax=ax, column=champ, cmap="viridis", linewidth=0.6, legend=True,
         legend_kwds={"label": titre, "shrink": 0.5})
ax.set_title("Traits de côte de Saint-Louis")
ax.set_axis_off()

fig.savefig(os.path.join(ROOT, "outputs", "figures", f"apercu_traits_de_cote{suffixe}.png"),
            dpi=200, bbox_inches="tight")
plt.show()

## 14. Générer le trait de référence (premier passage uniquement)

À exécuter après un premier passage **sans** trait de référence. La cellule construit le trait à partir des détections et l'enregistre dans `data/aoi`. Si un trait existe déjà, elle ne fait rien.

In [ ]:
from scipy.ndimage import median_filter

def generer_trait_reference(output, pas_y=25, pas_x=10, frequence_min=0.5,
                            ecart_pic=80, saut_coupure=150):
    # Pour chaque bande nord-sud de 25 m, on cherche la position la plus à l'ouest
    # où un trait est détecté sur au moins 50 % des dates : c'est la plage côté océan.
    # Les nuages au large changent d'une image à l'autre et sont donc ignorés.
    pts = [np.asarray(p)[:, :2] for p in output["shorelines"] if len(p) > 0]
    tous = np.vstack(pts)
    xmin, ymin = tous.min(axis=0)
    xmax, ymax = tous.max(axis=0)
    ny, nx = int((ymax - ymin) // pas_y) + 1, int((xmax - xmin) // pas_x) + 1
    presence = np.zeros((ny, nx))
    for p in pts:
        grille = np.zeros((ny, nx), bool)
        grille[((p[:, 1] - ymin) // pas_y).astype(int), ((p[:, 0] - xmin) // pas_x).astype(int)] = True
        presence += grille | np.roll(grille, 1, 1) | np.roll(grille, -1, 1)
    frequence = presence / len(pts)

    xs, ys = [], []
    for j in range(ny):
        k = np.where(frequence[j] >= frequence_min)[0]
        if len(k):
            xs.append(xmin + (k[0] + 0.5) * pas_x)
            ys.append(ymin + (j + 0.5) * pas_y)
    xs, ys = np.array(xs), np.array(ys)

    # Lissage : on retire les pics isolés (> 80 m de la médiane locale)
    garder = np.abs(xs - median_filter(xs, size=21, mode="nearest")) < ecart_pic
    xs, ys = xs[garder], ys[garder]

    # Coupure aux sauts réels (> 150 m), par exemple l'embouchure
    coupures = np.where(np.abs(np.diff(xs)) > saut_coupure)[0] + 1
    morceaux = [m for m in np.split(np.c_[xs, ys], coupures) if len(m) >= 4]
    return MultiLineString([LineString(m) for m in morceaux])

if os.path.exists(fn_ref):
    print("Un trait de référence existe déjà :", fn_ref)
else:
    trait = generer_trait_reference(output)
    gpd.GeoDataFrame({"nom": ["trait de référence (océan)"]}, geometry=[trait],
                     crs=CRS(settings["output_epsg"])).to_file(fn_ref, driver="GeoJSON")
    print(f"Trait de référence généré ({len(trait.geoms)} tronçons) : {fn_ref}")
    print("Vérifie-le dans QGIS, puis relance le notebook à partir de la cellule 10.")

## 15. Fusion des quatre périodes

Quand les quatre périodes ont été traitées, cette cellule les réunit dans un seul fichier `traits_de_cote_saint_louis_1984_2026.geojson`. Elle peut être lancée seule, après les cellules 1 à 5.

In [ ]:
import glob
import pandas as pd

fichiers = sorted(glob.glob(os.path.join(ROOT, "data", "processed", "traits_de_cote_saint_louis_p*.geojson")))
print("Périodes disponibles :", [os.path.basename(f) for f in fichiers])

attendu = len(PERIODES) if "PERIODES" in globals() else 4
if len(fichiers) < attendu:
    print(f"{attendu - len(fichiers)} période(s) manquante(s) : relance « Tout exécuter » (PERIODE = \"auto\").")
else:
    serie = gpd.GeoDataFrame(pd.concat([gpd.read_file(f) for f in fichiers], ignore_index=True),
                             crs=CRS(32628)).sort_values("date").reset_index(drop=True)
    fn_serie = os.path.join(ROOT, "data", "processed", "traits_de_cote_saint_louis_1984_2026.geojson")
    serie.to_file(fn_serie, driver="GeoJSON", encoding="utf-8")
    print(f"{len(serie)} traits de côte de {serie['annee'].min()} à {serie['annee'].max()} : {fn_serie}")
    print(serie.groupby("satname").size().rename("nombre de traits").to_string())

## Prochaine étape

Lancer les quatre périodes (une session Colab chacune, avec `PERIODE = "auto"`), puis la section 15 pour les fusionner.

Ensuite, notebook `02` : un trait de côte médian par année, puis calcul des taux d'évolution avec DSAS dans ArcGIS Pro (transects perpendiculaires à la côte, NSM, EPR, LRR).